In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("prathmeshzade/iiit5k-words", output_dir="data/iiit5k-words")

print("Path to dataset files:", path)

In [5]:
from paddleocr import PaddleOCR

In [7]:
model = PaddleOCR(
            ocr_version="PP-OCRv6",
            device="cpu",
            use_doc_orientation_classify=False,
            use_doc_unwarping=False,
            use_textline_orientation=False,
        )

Creating model: ('PP-OCRv6_medium_det', None, None)
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.


Using official model (PP-OCRv6_medium_det), the model files will be automatically downloaded and saved in `/home/ritesh/.paddlex/official_models/PP-OCRv6_medium_det`.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

RuntimeError: Engine 'paddle_static' is unavailable because dependency 'paddlepaddle' is not installed.

In [13]:
from paddleocr import TextDetection
det_model = TextDetection(model_name="PP-OCRv6_small_det", engine="transformers")

Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/ritesh/.paddlex/official_models/PP-OCRv6_small_det_safetensors`.


Loading weights:   0%|          | 0/271 [00:00<?, ?it/s]

In [14]:
det_model.paddlex_predictor

In [ ]:
from transformers import AutoImageProcessor, AutoModelForObjectDetection,PPOCRV6SmallDetForObjectDetection

In [22]:
image_processor = AutoImageProcessor.from_pretrained("PaddlePaddle/PP-OCRv6_tiny_det_safetensors")
model = AutoModelForObjectDetection.from_pretrained("PaddlePaddle/PP-OCRv6_tiny_det_safetensors")

preprocessor_config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.08k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.79MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/271 [00:00<?, ?it/s]

In [23]:
model

PPOCRV6SmallDetForObjectDetection(
  (model): PPOCRV6SmallDetModel(
    (backbone): PPLCNetV4Backbone(
      (encoder): PPLCNetV4Encoder(
        (convolution): PPLCNetV4LargeStem(
          (stem1): PPLCNetV4ConvLayer(
            (convolution): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
            (normalization): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (activation): ReLU()
            (lab): Identity()
          )
          (stem2a): PPLCNetV4ConvLayer(
            (convolution): Conv2d(16, 8, kernel_size=(2, 2), stride=(1, 1), bias=False)
            (normalization): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (activation): ReLU()
            (lab): Identity()
          )
          (stem2b): PPLCNetV4ConvLayer(
            (convolution): Conv2d(8, 16, kernel_size=(2, 2), stride=(1, 1), bias=False)
            (normalization): BatchNorm2d(16, eps=1e-0

In [ ]:
from io import BytesIO

from huggingface_hub.utils import httpx
from PIL import Image
from transformers import AutoImageProcessor, AutoModelForObjectDetection
from transformers.image_utils import load_image

model_path = "PaddlePaddle/PP-OCRv6_medium_det_safetensors"
model = AutoModelForObjectDetection.from_pretrained(model_path)
image_processor = AutoImageProcessor.from_pretrained(model_path)

image_url = "https://huggingface.co/datasets/hf-internal-testing/transformers-synthetic-assets/resolve/main/images/paddle_general_ocr_001.png"
image = load_image(image_url)
inputs = image_processor(images=image, return_tensors="pt").to(model.device)
outputs = model(**inputs)

results = image_processor.post_process_object_detection(
    outputs, 
    target_sizes=inputs["target_sizes"],
    threshold=0.2,
    box_threshold=0.45,
    max_candidates=3000,
    unclip_ratio=1.4,
)

for result in results:
    print(result)

ValueError: Using a `device_map`, `tp_plan`, `torch.device` context manager or setting `torch.set_default_device(device)` requires `accelerate`. You can install it with `pip install accelerate`

In [2]:
from paddleocr import TextRecognition
rec_model = TextRecognition(model_name="PP-OCRv6_medium_rec", engine="onnxruntime", device="cpu")

Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/ritesh/.paddlex/official_models/PP-OCRv6_medium_rec_onnx`.


In [1]:
image_path = "data/iiit5k-words/IIIT5K-Word_V3.0/IIIT5K/train/5197_3.png"

In [2]:
from PIL import Image
image = Image.open(image_path)

In [5]:
image.size

(163, 35)

In [8]:
output = det_model.predict(input=image_path, batch_size=1)

In [12]:
output[0].print()

{'res': {'input_path': 'data/iiit5k-words/IIIT5K-Word_V3.0/IIIT5K/train/5197_3.png', 'page_index': None, 'dt_polys': array([[[ 2,  2],
        ...,
        [ 2, 33]]], shape=(1, 4, 2), dtype=int16), 'dt_scores': [0.8137879985286768]}}


In [6]:
output = det_model.predict(input=image_path, batch_size=1)
for res in output:
    res.print()
    res.save_to_img(save_path="./output/")
    res.save_to_json(save_path="./output/res.json")

{'res': {'input_path': 'data/iiit5k-words/IIIT5K-Word_V3.0/IIIT5K/train/5197_3.png', 'page_index': None, 'dt_polys': array([[[ 2,  2],
        ...,
        [ 2, 33]]], shape=(1, 4, 2), dtype=int16), 'dt_scores': [0.8137879985286768]}}


In [7]:
output = rec_model.predict(input=image_path, batch_size=1)
for res in output:
    res.print()
    res.save_to_json(save_path="./output/rec.json")

{'res': {'input_path': 'data/iiit5k-words/IIIT5K-Word_V3.0/IIIT5K/train/5197_3.png', 'page_index': None, 'rec_text': 'COMING', 'rec_score': 0.9994008541107178}}


In [ ]:
from pymatreader import read_mat

testdata_mat = read_mat('testdata.mat')

testCharBound_mat = read_mat('testCharBound.mat')

testdata_mat

In [6]:
import onnx

model = onnx.load("tiny_det.onnx")
onnx.checker.check_model(model)

graph = model.graph

print("=== INPUTS ===")
for inp in graph.input:
    print("Name:", inp.name)
    print("Shape:", [
        d.dim_value if d.HasField("dim_value") else d.dim_param
        for d in inp.type.tensor_type.shape.dim
    ])
    print("Dtype:", onnx.TensorProto.DataType.Name(
        inp.type.tensor_type.elem_type
    ))

print("\n=== OUTPUTS ===")
for out in graph.output:
    print("Name:", out.name)
    print("Shape:", [
        d.dim_value if d.HasField("dim_value") else d.dim_param
        for d in out.type.tensor_type.shape.dim
    ])
    print("Dtype:", onnx.TensorProto.DataType.Name(
        out.type.tensor_type.elem_type
    ))

=== INPUTS ===
Name: x
Shape: ['DynamicDimension.0', 3, 'DynamicDimension.1', 'DynamicDimension.2']
Dtype: FLOAT

=== OUTPUTS ===
Name: fetch_name_0
Shape: ['ConvTranspose_459_o0__d0', 1, 'ConvTranspose_459_o0__d2', 'ConvTranspose_459_o0__d3']
Dtype: FLOAT


In [8]:
import numpy as np
import onnxruntime as ort

# Load ONNX model
session = ort.InferenceSession(
    "tiny_det.onnx",
    providers=["CPUExecutionProvider"]
)

# Create dummy input
x = np.random.rand(1, 3, 640, 640).astype(np.float32)

# Run inference
outputs = session.run(
    None,
    {"x": x}
)

# Inspect outputs
for i, output in enumerate(outputs):
    print(f"Output {i}")
    print("Shape:", output.shape)
    print("Dtype:", output.dtype)
    print("Min:", output.min())
    print("Max:", output.max())

Output 0
Shape: (1, 1, 640, 640)
Dtype: float32
Min: 0.0
Max: 0.0022080243


In [9]:
for h, w in [(320, 320), (640, 640), (640, 960)]:
    x = np.random.rand(1, 3, h, w).astype(np.float32)

    try:
        y = session.run(None, {"x": x})[0]
        print(f"Input: {x.shape} -> Output: {y.shape}")
    except Exception as e:
        print(f"Failed for {h}x{w}: {e}")

Input: (1, 3, 320, 320) -> Output: (1, 1, 320, 320)
Input: (1, 3, 640, 640) -> Output: (1, 1, 640, 640)
Input: (1, 3, 640, 960) -> Output: (1, 1, 640, 960)


In [1]:
import cv2

In [2]:
image_file = "data/iiit5k-words/IIIT5K-Word_V3.0/IIIT5K/train/5197_3.png"

In [3]:
img = cv2.imread(image_file)

In [4]:
img

array([[[ 94,  94,  94],
        [ 52,  52,  52],
        [ 58,  59,  55],
        ...,
        [102,  90,  96],
        [ 77,  63,  65],
        [ 68,  53,  51]],

       [[ 83,  83,  83],
        [ 62,  62,  62],
        [ 64,  63,  65],
        ...,
        [ 96,  93,  85],
        [ 77,  71,  60],
        [ 55,  48,  39]],

       [[ 39,  39,  39],
        [ 29,  29,  29],
        [ 37,  36,  40],
        ...,
        [ 54,  56,  44],
        [ 45,  45,  31],
        [ 33,  33,  21]],

       ...,

       [[  0,   0,   0],
        [  0,   0,   0],
        [  0,   0,   0],
        ...,
        [  7,   0,  33],
        [  3,   2,  22],
        [  7,   8,  22]],

       [[  0,   0,   0],
        [  0,   0,   0],
        [  0,   0,   0],
        ...,
        [  0,   0,  35],
        [  0,   0,  24],
        [  0,   0,  22]],

       [[  0,   0,   0],
        [  0,   0,   0],
        [  0,   0,   0],
        ...,
        [  2,   3,  59],
        [  0,   0,  47],
        [  0,   7,  54]]

In [1]:
import os
import pandas as pd
image_paths = []
image_texts = []

data_folder = "tmp_images"

for path in os.listdir(data_folder):
    image_paths.append(data_folder + "/" + path)
    image_texts.append(path.split("_")[1])


In [3]:
image_paths, image_texts

(['tmp_images/27_State_1.png', 'tmp_images/39_NOKIA_1.png'],
 ['State', 'NOKIA'])

In [4]:
df = pd.DataFrame({
    "ImgName": image_paths,
    "chars": image_texts
})

In [6]:
v = pd.read_csv("data/processed/val.csv")

In [7]:
v.head()

,ImgName,chars
0,data/iiit5k-words/IIIT5K-Word_V3.0/IIIT5K/trai...,DATO
1,data/iiit5k-words/IIIT5K-Word_V3.0/IIIT5K/trai...,Tall
2,data/iiit5k-words/IIIT5K-Word_V3.0/IIIT5K/trai...,AM
3,data/iiit5k-words/IIIT5K-Word_V3.0/IIIT5K/trai...,PAINS
4,data/iiit5k-words/IIIT5K-Word_V3.0/IIIT5K/trai...,THEYRE


In [8]:
df.head()

,ImgName,chars
0,tmp_images/27_State_1.png,State
1,tmp_images/39_NOKIA_1.png,NOKIA


In [9]:
df = pd.concat([df, v], ignore_index=True)

In [10]:
df.head()

,ImgName,chars
0,tmp_images/27_State_1.png,State
1,tmp_images/39_NOKIA_1.png,NOKIA
2,data/iiit5k-words/IIIT5K-Word_V3.0/IIIT5K/trai...,DATO
3,data/iiit5k-words/IIIT5K-Word_V3.0/IIIT5K/trai...,Tall
4,data/iiit5k-words/IIIT5K-Word_V3.0/IIIT5K/trai...,AM


In [1]:
import torch

In [2]:
xx = torch.load("data/processed/vocab.pt")

In [3]:
len(xx['vocab'])

62

In [2]:
import torch

from src.rec_model.model import CRNN

state_dict = torch.load(
    "crnn_best2.pt",
    map_location="cpu",
    weights_only=True
)
vocab = torch.load("data/processed/vocab.pt")['vocab']

model = CRNN(num_classes=len(vocab) + 1, height=32)
model.load_state_dict(state_dict['model'])
model.eval()

dummy = torch.randn(1, 1, 32, 128)

torch.onnx.export(
    model,
    (dummy,),
    "crnn_fp32.onnx",
    input_names=["images"],
    output_names=["logits"],
    opset_version=17,
    dynamo=False,
    verify=True,
)

/home/ritesh/miniconda3/envs/ocr/lib/python3.10/site-packages/torch/onnx/symbolic_opset9.py:4277: UserWarning: Exporting a model to ONNX with a batch_size other than 1, with a variable length with LSTM can cause an error when running the ONNX model with a different batch size. Make sure to save the model with a batch size of 1, or define the initial states (h0/c0) as inputs of the model. 
  warnings.warn(


In [3]:
import torch
import onnx
import onnxruntime as ort
import numpy as np

# Check ONNX graph
onnx_model = onnx.load("crnn_fp32.onnx")
onnx.checker.check_model(onnx_model)

# ONNX Runtime
session = ort.InferenceSession(
    "crnn_fp32.onnx",
    providers=["CPUExecutionProvider"]
)

x = np.random.randn(1, 1, 32, 128).astype(np.float32)

with torch.no_grad():
    torch_output = model(torch.from_numpy(x)).numpy()

onnx_output = session.run(
    ["logits"],
    {"images": x}
)[0]

print("PyTorch shape:", torch_output.shape)
print("ONNX shape:", onnx_output.shape)

print("Max absolute difference:",
      np.max(np.abs(torch_output - onnx_output)))

print("Outputs close:",
      np.allclose(
          torch_output,
          onnx_output,
          rtol=1e-3,
          atol=1e-4
      ))

PyTorch shape: (32, 1, 63)
ONNX shape: (32, 1, 63)
Max absolute difference: 7.1525574e-06
Outputs close: True


2026-10-15 16:26:47.508765473 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


In [8]:
import argparse
import numpy as np
import onnxruntime
import time
from onnxruntime.quantization import QuantFormat, QuantType, quantize_static, quantize_dynamic


def benchmark(model_path):
    session = onnxruntime.InferenceSession(model_path)
    input_name = session.get_inputs()[0].name

    total = 0.0
    runs = 10
    input_data = np.zeros((1, 1, 32, 128), np.float32)
    # Warming up
    _ = session.run([], {input_name: input_data})
    for i in range(runs):
        start = time.perf_counter()
        _ = session.run([], {input_name: input_data})
        end = (time.perf_counter() - start) * 1000
        total += end
        print(f"{end:.2f}ms")
    total /= runs
    print(f"Avg: {total:.2f}ms")


In [9]:
input_model_path = "crnn_fp32.onnx"
output_model_path = "crnn_int8.onnx"

In [10]:
benchmark(input_model_path)

4.69ms
6.61ms
6.95ms
7.31ms
5.57ms
6.35ms
9.11ms
5.78ms
7.15ms
6.98ms
Avg: 6.65ms


In [ ]:
quantize_static(
        input_model_path,
        output_model_path,
        quant_format=QuantFormat.QDQ,
        per_channel=False,
        weight_type=QuantType.QInt8,
    )

In [15]:
quantize_dynamic(
        input_model_path,
        output_model_path,
op_types_to_quantize=["LSTM", "MatMul"],
        weight_type=QuantType.QInt8,
    )

In [16]:
benchmark(output_model_path)

4.40ms
5.77ms
19.35ms
4.77ms
4.43ms
3.99ms
4.86ms
6.01ms
4.95ms
4.45ms
Avg: 6.30ms
